In [1]:
import pandas as pd
import numpy as np

athletes = pd.read_csv('../data/raw/olympic_athletes.csv')
hosts = pd.read_csv('../data/raw/olympic_hosts.csv')
medals = pd.read_csv('../data/raw/olympic_medals.csv')
results = pd.read_csv('../data/raw/olympic_results.csv')


print("ATHLETES")
athletes.info()
athletes.head()

print("MEDALS")
medals.info()

print("RESULTS")
results.info()

ATHLETES
<class 'pandas.DataFrame'>
RangeIndex: 75904 entries, 0 to 75903
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   athlete_url           75904 non-null  str    
 1   athlete_full_name     75904 non-null  str    
 2   games_participations  75904 non-null  int64  
 3   first_game            75882 non-null  str    
 4   athlete_year_birth    73448 non-null  float64
 5   athlete_medals        15352 non-null  str    
 6   bio                   22842 non-null  str    
dtypes: float64(1), int64(1), str(5)
memory usage: 4.1 MB
MEDALS
<class 'pandas.DataFrame'>
RangeIndex: 21697 entries, 0 to 21696
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   discipline_title       21697 non-null  str  
 1   slug_game              21697 non-null  str  
 2   event_title            21697 non-null  str  
 3   event_gender     

In [ ]:
# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import pandas as pd
import numpy as np

# ============================================================
# 2. CARGA DE DATASETS
# Se cargan los archivos originales desde la carpeta raw
# ============================================================

athletes = pd.read_csv('../data/raw/olympic_athletes.csv')
hosts = pd.read_csv('../data/raw/olympic_hosts.csv')
medals = pd.read_csv('../data/raw/olympic_medals.csv')
results = pd.read_csv('../data/raw/olympic_results.csv')

# ============================================================
# 3. NORMALIZACIÓN DE NOMBRES DE COLUMNAS
# Se transforman a minúsculas, sin espacios y formato uniforme
# ============================================================

for df in [athletes, hosts, medals, results]:
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(' ', '_')
    )

# ============================================================
# 4. DETECCIÓN Y ELIMINACIÓN DE DUPLICADOS
# ============================================================

print("Duplicados athletes:", athletes.duplicated().sum())
print("Duplicados hosts:", hosts.duplicated().sum())
print("Duplicados medals:", medals.duplicated().sum())
print("Duplicados results:", results.duplicated().sum())

athletes = athletes.drop_duplicates()
hosts = hosts.drop_duplicates()
medals = medals.drop_duplicates()
results = results.drop_duplicates()

# ============================================================
# 5. LIMPIEZA DE TABLA HOSTS (SEDES OLÍMPICAS)
# ============================================================

hosts_clean = hosts.copy()

# Convertimos fechas a formato datetime
hosts_clean['game_start_date'] = pd.to_datetime(hosts_clean['game_start_date'], errors='coerce')
hosts_clean['game_end_date'] = pd.to_datetime(hosts_clean['game_end_date'], errors='coerce')

# Renombramos columnas para estandarizar
hosts_clean = hosts_clean.rename(columns={
    'game_slug': 'slug_game',
    'game_location': 'host_country',
    'game_name': 'olympic_game',
    'game_season': 'season',
    'game_year': 'year'
})

# Seleccionamos columnas relevantes
hosts_clean = hosts_clean[
    ['slug_game', 'olympic_game', 'year', 'season', 'host_country', 'game_start_date', 'game_end_date']
]

# ============================================================
# 6. LIMPIEZA DE TABLA ATHLETES (ATLETAS)
# ============================================================

athletes_clean = athletes.copy()

# Normalizamos nombres de atletas
athletes_clean['athlete_full_name'] = athletes_clean['athlete_full_name'].str.strip().str.title()

# Convertimos año de nacimiento a numérico
athletes_clean['athlete_year_birth'] = pd.to_numeric(
    athletes_clean['athlete_year_birth'], errors='coerce'
)

# Renombramos columnas
athletes_clean = athletes_clean.rename(columns={
    'games_participations': 'total_games_participations',
    'athlete_year_birth': 'birth_year'
})

# Seleccionamos columnas útiles
athletes_clean = athletes_clean[
    ['athlete_url', 'athlete_full_name', 'total_games_participations', 'first_game', 'birth_year']
]

# ============================================================
# 7. LIMPIEZA DE TABLA RESULTS (RESULTADOS)
# ============================================================

results_clean = results.copy()

# Columnas de texto a limpiar
text_cols = [
    'discipline_title', 'event_title', 'slug_game', 'participant_type',
    'medal_type', 'country_name', 'country_code',
    'country_3_letter_code', 'athlete_full_name'
]

# Eliminamos espacios y aseguramos tipo string
for col in text_cols:
    if col in results_clean.columns:
        results_clean[col] = results_clean[col].astype('string').str.strip()

# Reemplazamos valores nulos en medallas
results_clean['medal_type'] = results_clean['medal_type'].fillna('NO_MEDAL')

# Convertimos posición a numérico
results_clean['rank_position'] = pd.to_numeric(results_clean['rank_position'], errors='coerce')

# ============================================================
# 8. UNIÓN DE DATASETS
# ============================================================

# Unión con sedes olímpicas
df = results_clean.merge(
    hosts_clean,
    on='slug_game',
    how='left'
)

# Unión con datos de atletas
df = df.merge(
    athletes_clean,
    on=['athlete_url', 'athlete_full_name'],
    how='left'
)

# ============================================================
# 9. CREACIÓN DE VARIABLES NUEVAS
# ============================================================

# Variable binaria: tiene medalla o no
df['has_medal'] = np.where(df['medal_type'] != 'NO_MEDAL', 1, 0)

# Variables por tipo de medalla
df['gold'] = np.where(df['medal_type'] == 'GOLD', 1, 0)
df['silver'] = np.where(df['medal_type'] == 'SILVER', 1, 0)
df['bronze'] = np.where(df['medal_type'] == 'BRONZE', 1, 0)

# Total de medallas
df['total_medals'] = df[['gold', 'silver', 'bronze']].sum(axis=1)

# Normalización de texto
df['country_name'] = df['country_name'].str.title()
df['discipline_title'] = df['discipline_title'].str.title()
df['event_title'] = df['event_title'].str.title()

# ============================================================
# 10. CREACIÓN DE VARIABLE EDAD
# ============================================================

df['athlete_age'] = df['year'] - df['birth_year']

# Eliminamos edades irreales
df.loc[(df['athlete_age'] < 10) | (df['athlete_age'] > 80), 'athlete_age'] = np.nan

# ============================================================
# 11. SELECCIÓN DE VARIABLES FINALES
# ============================================================

final_columns = [
    'year',
    'season',
    'olympic_game',
    'host_country',
    'discipline_title',
    'event_title',
    'participant_type',
    'country_name',
    'country_code',
    'country_3_letter_code',
    'athlete_full_name',
    'athlete_url',
    'birth_year',
    'athlete_age',
    'total_games_participations',
    'rank_position',
    'medal_type',
    'has_medal',
    'gold',
    'silver',
    'bronze',
    'total_medals'
]

df_clean = df[final_columns].copy()

# ============================================================
# 12. VALIDACIÓN FINAL
# ============================================================

print("Filas y columnas:", df_clean.shape)

df_clean.info()

print("\nValores nulos:")
print(df_clean.isnull().sum().sort_values(ascending=False))

print("\nDistribución de medallas:")
print(df_clean['medal_type'].value_counts())

print("\nResumen de medallas:")
print(df_clean[['gold', 'silver', 'bronze', 'total_medals']].sum())

# ============================================================
# 13. EXPORTACIÓN DEL DATASET LIMPIO
# ============================================================

df_clean.to_csv('../data/processed/olympics_clean.csv', index=False, encoding='utf-8')

# Verificación de guardado
test = pd.read_csv('../data/processed/olympics_clean.csv')
test.head()